# Xếp hạng bốn lựa chọn ViMMRC 2.0

Mục tiêu: huấn luyện một mô hình CPU nhỏ chọn A–D từ câu hỏi, passage và bốn lựa chọn. Đây là nhánh **trắc nghiệm**, không phải mô hình trả lời tự do. Dữ liệu ViMMRC có điều kiện truy cập; notebook không tải hay công bố dữ liệu.

## 1. Chuẩn bị

Cài `pip install -r requirements_choice.txt` trong thư mục module. Các cell tìm module khi chạy từ thư mục module hoặc gốc repository.

In [ ]:
from pathlib import Path
import json, sys, subprocess
candidates = [Path.cwd(), Path.cwd() / 'mrc-model_Huy',
              Path.cwd() / 'vimmrc-qa-voice' / 'mrc-model_Huy']
WORK_DIR = next((p for p in candidates if (p / 'choice_ranker.py').exists()), None)
assert WORK_DIR is not None, 'Run from repository root or mrc-model_Huy'
sys.path.insert(0, str(WORK_DIR.resolve()))
from choice_ranker import ChoiceRanker, lexical_baseline
from mrc_data import load_examples
print('Module:', WORK_DIR.resolve())

## 2. Schema và baseline không huấn luyện

ViMMRC lưu một bài đọc cùng mảng câu hỏi, lựa chọn và nhãn A–D. Baseline lexical chọn lựa chọn có tỷ lệ từ xuất hiện trong passage cao nhất. Dữ liệu ví dụ là tự tạo.

In [ ]:
sample = load_examples(WORK_DIR / 'examples' / 'sample_vimmrc.json')
row = sample[0]
print('Gold:', row['answer_letter'], row['answer'])
print('Lexical:', lexical_baseline(row['question'], row['passage'], row['options']))

## 3. Quy tắc đánh giá

Chỉ tập **train** được dùng để học TF‑IDF và logistic regression. Chia bài đọc trong train thành inner train/validation theo nhóm bài để chọn `C` trên {0.1, 1, 10}. Sau đó fit lại trên toàn train và đánh giá **dev đúng một lần**. Script không mở `test_vimmrc.json`. So sánh cùng tập dev với lexical và random có seed.

In [ ]:
DATA_DIR = WORK_DIR.parent / 'data'
TRAIN = DATA_DIR / 'train_vimmrc.json'
DEV = DATA_DIR / 'dev_vimmrc.json'
print('Train/dev available:', TRAIN.exists() and DEV.exists())
if TRAIN.exists() and DEV.exists():
    print('Questions:', {'train': len(load_examples(TRAIN)), 'dev': len(load_examples(DEV))})

## 4. Huấn luyện CPU

Đặt `RUN_TRAINING=True` để tái tạo checkpoint và report. Một lần chạy trên CPU mất khoảng một phút tùy máy; không tải mô hình lớn.

In [ ]:
RUN_TRAINING = False
CHECKPOINT = WORK_DIR / 'checkpoints' / 'choice_ranker'
if RUN_TRAINING:
    assert TRAIN.exists() and DEV.exists(), 'Licensed train/dev files are required'
    subprocess.run([sys.executable, str(WORK_DIR / 'train_choice_ranker.py'),
                    '--train', str(TRAIN), '--dev', str(DEV),
                    '--output-dir', str(CHECKPOINT), '--seed', '42'], check=True)
else:
    print('Training disabled; set RUN_TRAINING=True to reproduce the experiment.')

## 5. Đọc kết quả và suy luận

Các điểm là độ chính xác chọn nhãn trong bốn lựa chọn; baseline ngẫu nhiên danh nghĩa 25%. Artifact `ranker.joblib` phải là tệp do bạn tạo hoặc tin cậy.

In [ ]:
report_file = CHECKPOINT / 'report.json'
if report_file.exists():
    report = json.loads(report_file.read_text(encoding='utf-8'))
    for key in ('inner_valid_accuracy', 'dev_ranker_accuracy',
                'dev_lexical_accuracy', 'dev_seeded_random_accuracy'):
        print(f'{key}: {report[key]:.4f}')
    assert report['test_evaluated'] is False
else:
    print('No report yet; run step 4 after obtaining licensed train/dev data.')

In [ ]:
if (CHECKPOINT / 'ranker.joblib').exists():
    ranker = ChoiceRanker.load(CHECKPOINT)
    print(ranker.predict(row['question'], row['passage'], row['options']))
    print('Answer:', ranker.answer_question(row['question'], row['passage'], row['options']))
else:
    print('Train the choice ranker first to enable inference.')

## 6. Giới hạn

Mô hình luôn cần bốn lựa chọn được cung cấp; không áp dụng cho câu hỏi không có phương án. Độ chính xác dev đo trên passage gốc. Khi retrieval chọn sai passage, kết quả có thể thấp hơn. Không dùng dev để chỉnh đặc trưng sau lần đánh giá này; test tiếp tục để riêng.